# 004 · while Loops

> **Syntax › Control Flow** · Python 3.14

`while` repeats a block as long as a condition is truthy. It works like C#'s `while`, but Python has **no `do`/`while`** — you emulate it with `while True` and `break`.

**In this notebook you will learn:**

- the `while` syntax and how it compares to C#
- how to emulate `do { } while (cond);`
- writing intentional infinite loops safely
- using the walrus operator `:=` in a `while` condition
- sentinel loops, including the two-argument form of `iter()`

## C# mental model

| C# | Python |
|---|---|
| `while (cond) { ... }` | `while cond:` |
| `do { ... } while (cond);` | `while True:` ... `if not cond: break` |
| `while (true) { ... }` | `while True:` |
| `while ((line = reader.ReadLine()) != null)` | `while (line := f.readline()):` |

## Basic `while`

```csharp
// C#
int n = 3;
while (n > 0)
{
    Console.WriteLine(n);
    n--;
}
```

Python has no `++`/`--` operators — use `+= 1` / `-= 1`:

In [ ]:
n = 3
while n > 0:
    print(n)
    n -= 1                            # no n-- in Python
print("liftoff!")

The condition uses truthiness *(see Data Types › 004 · Booleans and Truthiness)*, so "loop while the list is non-empty" is just `while items:`:

In [ ]:
queue = ["job-1", "job-2", "job-3"]
while queue:                          # C#: while (queue.Count > 0)
    job = queue.pop(0)
    print("processing", job)
print("queue empty:", queue)

A classic algorithmic `while` — the number of iterations isn't known in advance, which is when `while` is the right tool instead of `for`:

In [ ]:
def gcd(a: int, b: int) -> int:
    while b:                          # while b != 0
        a, b = b, a % b               # tuple assignment: no temp variable needed
    return a

print(gcd(48, 18))

## No `do`/`while` — emulate it

A C# `do`/`while` runs the body at least once and checks the condition at the end. Python has no such statement; the idiom is an infinite loop with a `break` at the bottom.

```csharp
// C#
int attempts = 0;
do
{
    attempts++;
    Console.WriteLine($"attempt {attempts}");
} while (attempts < 3);
```

In [ ]:
attempts = 0
while True:
    attempts += 1
    print(f"attempt {attempts}")
    if not attempts < 3:              # the inverted do/while condition
        break

A realistic example — keep "rolling" until we get a six. The body must run at least once, so this is a natural `do`/`while`:

In [ ]:
import random

random.seed(7)                        # deterministic output for the notebook
rolls = 0
while True:
    roll = random.randint(1, 6)
    rolls += 1
    print("rolled", roll)
    if roll == 6:
        break
print(f"got a six after {rolls} roll(s)")

## Infinite loops

`while True:` is the standard way to write an intentionally infinite loop — servers, event loops, REPL-style prompts. Always make sure there is a reachable `break`, `return` or exception.

A typical interactive loop looks like this (shown as text — `input()` can't run in this executed notebook):

```python
while True:
    command = input("> ").strip()
    if command == "quit":
        break
    print(f"you typed {command!r}")
```

⚠️ **Gotcha:** a runaway `while` loop in Jupyter keeps the kernel busy. Use **Kernel › Interrupt** (or press `I`, `I` in command mode) to stop it — this raises `KeyboardInterrupt` in your code, just like Ctrl+C in a terminal.

A useful defensive pattern is to cap the number of iterations:

In [ ]:
MAX_ITERATIONS = 1_000
value, steps = 27, 0

while value != 1:                     # Collatz sequence — terminates, but nobody has proved it always does!
    value = value // 2 if value % 2 == 0 else 3 * value + 1
    steps += 1
    if steps >= MAX_ITERATIONS:
        print("giving up")
        break
print(f"reached 1 after {steps} steps")

## The walrus operator in `while`

C# lets you assign inside a loop condition: `while ((line = reader.ReadLine()) != null)`. Python's `=` is a statement and can't appear there, but the **assignment expression** `:=` (the "walrus", 3.8+) can. It removes the duplicated "read before the loop and again at the end" pattern.

```csharp
// C#
string? line;
while ((line = reader.ReadLine()) != null)
    Console.WriteLine(line);
```

In [ ]:
import io

reader = io.StringIO("first line\nsecond line\nthird line\n")

# Without walrus: the read is duplicated
line = reader.readline()
while line:
    print(line.rstrip())
    line = reader.readline()

In [ ]:
reader = io.StringIO("first line\nsecond line\nthird line\n")

# With walrus: assign and test in the condition
while (line := reader.readline()):
    print(line.rstrip())

Reading in fixed-size chunks is another common case:

In [ ]:
data = io.BytesIO(b"abcdefghij")
while chunk := data.read(4):          # b"" (empty bytes) is falsy -> loop ends
    print(chunk)

⚠️ **Gotcha:** `:=` has very low precedence. `while n := f() > 0` assigns the **boolean** `f() > 0` to `n`. Parenthesise: `while (n := f()) > 0`.

In [ ]:
values = iter([3, 2, 1, 0, 5])

while (n := next(values)) > 0:        # correct: n is the number
    print("n =", n)

values = iter([3, 2, 1, 0, 5])
while n := next(values) > 0:          # wrong: n is True, not the number
    print("n =", n)

## Sentinel loops

A **sentinel** is a special value that signals "stop" — e.g. an empty line, `None`, or `"quit"`. The walrus pattern above is one sentinel loop. Python also has a neat built-in for this: `iter(callable, sentinel)` calls `callable()` repeatedly until it returns `sentinel`, so you can use a plain `for` loop.

In [ ]:
commands = ["start", "status", "quit", "never seen"]
source = iter(commands)

def next_command() -> str:
    return next(source)

for command in iter(next_command, "quit"):   # stop when "quit" is returned
    print("handling", command)

In [ ]:
from functools import partial

reader = io.BytesIO(b"0123456789")
for block in iter(partial(reader.read, 3), b""):   # read 3 bytes until empty
    print(block)

When the sentinel must be distinguishable from **every** real value (including `None`), create a unique object:

In [ ]:
_MISSING = object()                   # a unique sentinel, nothing else is identical to it

stack = [1, None, 2]

def pop_or_missing():
    return stack.pop() if stack else _MISSING

while (item := pop_or_missing()) is not _MISSING:
    print("popped", item)             # None is a valid item here

## `while` vs `for`

- Use `for` when iterating over a collection or a known `range` *(see Control Flow › 003 · for Loops and range)*.
- Use `while` when the number of iterations depends on a condition that changes inside the loop.
- `while` also supports `else`, `break` and `continue` *(see Control Flow › 005 · break, continue and Loop else)*.

## Summary

| Task | C# | Python |
|---|---|---|
| Loop while true | `while (cond) { }` | `while cond:` |
| Increment | `i++` | `i += 1` |
| Do/while | `do { } while (cond);` | `while True:` ... `if not cond: break` |
| Infinite loop | `while (true)` / `for (;;)` | `while True:` |
| Assign in condition | `while ((x = F()) != null)` | `while (x := f()) is not None:` |
| Loop until sentinel | manual | `for x in iter(func, sentinel):` |

## Exercises

1. Using `while`, compute how many times you can halve 1000 (integer division) before it reaches 0.
2. Emulate a `do`/`while`: starting from `balance = 100`, repeatedly subtract 30 and print the balance, and stop once the balance would go negative. The body must run at least once.
3. Given `tokens = iter(["a", "b", "", "c"])`, use a walrus `while` loop to print tokens until an empty string is encountered. Then rewrite it with `iter(callable, sentinel)`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
n, count = 1000, 0
while n > 0:
    n //= 2
    count += 1
print(count)    # 10

# 2
balance = 100
while True:
    balance -= 30
    print(balance)
    if balance - 30 < 0:
        break

# 3
tokens = iter(["a", "b", "", "c"])
while (token := next(tokens)):
    print(token)

tokens = iter(["a", "b", "", "c"])
for token in iter(lambda: next(tokens), ""):
    print(token)
```

</details>